# 05. マルチターン Chat と Function Calling（ツール連携）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/ja/05_chat_and_function_calling.ipynb)

`google-genai` SDK は、ステートレスなモデル呼び出しを「状態とツールを備えた実用的なアシスタント」へと進化させる2つの標準プリミティブを提供しています。会話履歴を自動管理する **`client.chats.create()`** と、Python の型ヒントおよび docstring からスキーマを自動抽出し、ローカル関数の実行から結果の再投入までを自動化する **自動 Function Calling（`tools=[fn]`）** です。

![図5-1: google-genai SDK における Python 関数の自動 Function Calling ループ](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_function_calling_loop_ja.png)

*図5-1: google-genai SDK における Python 関数の自動 Function Calling ループ*

---

## 1. クライアント初期化とマルチターン会話（`client.chats.create`）

セルの先頭で `MODEL = "gemini-3.8-flash"` を定義し、Google Colab のシークレット（`userdata.get`）から `GEMINI_API_KEY` を読み込んで共通の `genai.Client` を初期化します。

In [ ]:
!pip install -q google-genai

In [ ]:
MODEL = "gemini-3.8-flash"

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

単発の `client.models.generate_content()` とは異なり、**`client.chats.create()`** はセッション内部で `list[types.Content]` 形式の会話履歴を保持します。`chat.send_message()` を呼ぶたびに `Content(role="user")` とモデルの応答 `Content(role="model")`（および途中のツール呼び出し履歴）が自動的に追記されます。さらにマルチターン会話が長くなると、毎ターン送信される共通プレフィックスに対して Gemini 3.8 の **暗黙的コンテキストキャッシュ（Implicit Context Caching）** が追加設定なしで自動的に働き、レイテンシ（TTFT）と入力トークンコストが抑えられます。

![図5-2: chats.create() による会話履歴の蓄積とツール連携セッションの構造](https://examples.drawlib.com/google-genai/images/05_chat_and_function_calling/ch05_chat_state_context_ja.png)

*図5-2: chats.create() による会話履歴の蓄積とツール連携セッションの構造*

`MODEL`（`gemini-3.8-flash`）でチャットセッションを作成し、短い2ターンのメッセージを送信したあと、`chat.get_history()` に蓄積された `user` と `model` のターンを確認してみましょう。

In [ ]:
chat = client.chats.create(model=MODEL)

reply_1 = chat.send_message("私の好きなプログラミング言語はPythonです。")
print("Turn 1:", reply_1.text)

reply_2 = chat.send_message("私の好きな言語は何ですか？1文で答えてください。")
print("Turn 2:", reply_2.text)

print("\n--- 蓄積されたセッション履歴 ---")
for idx, turn in enumerate(chat.get_history(), start=1):
    preview = " ".join(
        p.text.strip() for p in (turn.parts or []) if p.text and not p.thought
    )
    print(f"[{idx}] role={turn.role:<5} | {preview[:80]}")

---

## 2. Python 関数の自動 Function Calling とツール付きチャット

外部の監視メトリクスや社内 API を Gemini に連携させるには、型ヒント（`region: str -> dict[str, float]`）と **Google スタイルの docstring**（`Args:` / `Returns:`）を記述した通常の Python 関数を **`types.GenerateContentConfig(tools=[...])`** にそのまま渡します。SDK が関数シグネチャから JSON スキーマを自動生成します。

まずは単発の `client.models.generate_content(model=MODEL, ...)` で2つのリージョンのレイテンシを比較してみましょう。`genai.Client` が各リージョンに対してローカルの `get_region_latency()` を自動実行し、`FunctionResponse` をモデルへ返送して実測値に基づく比較結果を出力します。

In [ ]:
from google.genai import types

REGION_LATENCY_MS: dict[str, dict[str, float]] = {
    "asia-northeast1": {"p50_ms": 14.2, "p95_ms": 31.5},
    "us-central1": {"p50_ms": 24.5, "p95_ms": 58.4},
    "europe-west1": {"p50_ms": 38.6, "p95_ms": 84.2},
}


def get_region_latency(region: str) -> dict[str, float]:
    """指定されたクラウドリージョンの最新 API レイテンシ（p50 / p95 ミリ秒）を取得します。

    Args:
        region: 対象リージョン ID（'asia-northeast1', 'us-central1', 'europe-west1'）。

    Returns:
        'p50_ms'（中央値）と 'p95_ms'（95パーセンタイル）を含む辞書。
    """
    return REGION_LATENCY_MS.get(region, {"p50_ms": -1.0, "p95_ms": -1.0})


response = client.models.generate_content(
    model=MODEL,
    contents="asia-northeast1 と us-central1 のレイテンシを比較してください。",
    config=types.GenerateContentConfig(
        tools=[get_region_latency],
        temperature=0.1,
    ),
)
print(response.text)

続いて、これまでの2つの機能を組み合わせて **`client.chats.create(model=MODEL, config=...)`** に `tools=[get_region_latency]` を渡します。フォローアップ質問を重ねても過去の文脈を維持したまま必要に応じて追加のツール呼び出しを実行し、すべての `FunctionCall` と `FunctionResponse` が `tool_chat.get_history()` に記録されます。

In [ ]:
tool_chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        tools=[get_region_latency],
        temperature=0.1,
    ),
)

turn_1 = tool_chat.send_message("asia-northeast1 と us-central1 のレイテンシを確認してください。")
print("Turn 1:\n", turn_1.text)

turn_2 = tool_chat.send_message("europe-west1 も確認して、最速のリージョンを1文で答えてください。")
print("\nTurn 2:\n", turn_2.text)

print("\n--- 記録されたツール呼び出しトレース ---")
for turn in tool_chat.get_history():
    for part in turn.parts or []:
        if part.function_call:
            fc = part.function_call
            print(f"-> FunctionCall: {fc.name}({dict(fc.args or {})})")
        elif part.function_response:
            print(f"<- FunctionResponse: {part.function_response.response}")

> **まとめ**: ノートブック先頭で `MODEL = "gemini-3.8-flash"` を定義し、マルチターン履歴を自動管理する `client.chats.create(model=MODEL)` と、型付き Python 関数を `tools=[fn]` に渡すだけの自動 Function Calling を組み合わせることで、単発の `generate_content()` から複数ターンのツール付きチャットまでボイラープレートなしで実装できます。